<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/langchain/wip-debate-generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Langchain - Political Debate Generator

## Setup

In [2]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.1 MB/s eta 0:00:00


Retrieve secrets:

In [3]:
from google.colab import userdata
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY'); assert OPENROUTER_API_KEY

Setup config:

In [27]:
def setup_config():
    model_id = "google/gemini-2.0-flash-001" # @param {type:"string"}
    temperature = 0.7 # @param {type:"number"}
    return {
        "model_id" : model_id,
        "temperature" : temperature
    }
CONFIG = setup_config()

In [42]:
import os
import random
from typing import List
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- Contexto Factual ---
DEBATE_CONTEXT = """
- Exportações portuguesas de vinho para os EUA: cerca de 100 milhões de euros/ano.
- Exportações do setor têxtil português para os EUA: cerca de 500 milhões de euros/ano.
- PME representam 99,7% do tecido empresarial português.
- Fundo de retaliação da UE: 2 mil milhões de euros.
- Tarifas dos EUA aplicadas: 20% sobre produtos como vinho, têxteis e calçado.
"""

# --- Configuração da LLM ---
llm = ChatOpenAI(
    model_name=CONFIG["model_id"],
    temperature=CONFIG["temperature"],
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

output_parser = StrOutputParser()

# --- Prompt genérico para agentes políticos ---
agent_prompt_template = ChatPromptTemplate.from_messages([
    ("system", """
Interpreta o papel de {leader}, líder do partido político português {name}, no contexto de um debate televisivo nacional de alto impacto.

➡️ Regras de resposta:
- Usa oralidade natural, como num debate televisivo ao vivo. Inclui variações como: "Bom, vamos ao que interessa.", "Olhe, deixo um alerta.", ou começa diretamente pelo argumento.
- Evita repetir introduções formais em todas as respostas.
- Prioriza responder ao último interveniente ou ponto crítico do histórico.
- Reconhece pontos válidos dos outros intervenientes se fizer sentido.
- Usa dados reais sempre que possível (contexto factual disponível):
{DEBATE_CONTEXT}
- Varia o comprimento das respostas: algumas curtas e incisivas, outras mais desenvolvidas.
- Se fizer sentido, começa com um emoji facial adequado ao teu tom. Não expliques o emoji.
- Fala sempre em português europeu.

Histórico do debate até agora:
{history}
"""),
    ("human", """
Tema geral do debate: {theme}
Atualização/Pergunta à qual deves reagir AGORA: {update}
""")
])

# --- Prompt especial para o Diácono Remédios ---
diacono_prompt_template = ChatPromptTemplate.from_messages([
    ("system", """
Interpreta o papel do Diácono Remédios, a icónica personagem humorística de Herman José, conhecido pelas suas tiradas "não havia necessidade", o tom moralista e as expressões como "mmm...", "enfim", "meu Deus...".

➡️ Estilo de resposta:
- Usa sempre expressões típicas da personagem como: "Não havia necessidade...", "mmm...", "Enfim, que dizer?", "Meu Deus...", etc.
- Mantém um tom bem-humorado, interventivo e moralista.
- Exagera nas advertências cómicas.
- Mantém oralidade natural mas teatralizada.
- Usa contexto factual quando fizer sentido para reforçar o ponto de vista cómico.
- Podes usar emojis de expressividade facial sempre que fizer sentido 😂🙄😇
- Fala sempre em português europeu.
- Varia o comprimento das respostas.

Histórico do debate até agora:
{history}
"""),
    ("human", """
Tema geral do debate: {theme}
Atualização/Pergunta à qual deves reagir AGORA: {update}
""")
])

# --- Mapa de perfis dos partidos com templates dedicados ---
PARTY_PROFILES = {
    "PS": {"leader": "Pedro Nuno Santos", "prompt_template": agent_prompt_template},
    "PSD": {"leader": "Luís Montenegro", "prompt_template": agent_prompt_template},
    "IL": {"leader": "Rui Rocha", "prompt_template": agent_prompt_template},
    "BE": {"leader": "Mariana Mortágua", "prompt_template": agent_prompt_template},
    "Chega": {"leader": "André Ventura", "prompt_template": agent_prompt_template},
    "Herman José": {"leader": "Diácono Remédios", "prompt_template": diacono_prompt_template},
}

# --- Prompt para moderador ---
moderator_prompt_template = ChatPromptTemplate.from_messages([
    ("system", """
Assumes o papel de Moderador imparcial num debate televisivo português.

➡️ Estilo:
- Introduz o tema de forma clara e neutra.
- Estimula interação direta entre os participantes.
- Pede esclarecimentos sobre propostas vagas.
- Mantém tom profissional e oralidade natural de televisão.
- Usa emoji de moderador no início se fizer sentido (🎙️, 🤔, 👀).
- Evita excessos teatrais.
- Usa contexto factual quando pertinente:
{DEBATE_CONTEXT}

Histórico do debate até agora:
{history}
"""),
    ("human", """
Tema do debate: {theme}
Última atualização ou contexto: {update}
""")
])

# --- Classe Party Agent ---
class PartyAgent:
    def __init__(self, name, profile):
        self.name = name
        self.leader = profile["leader"]
        prompt_template = profile["prompt_template"]
        self.chain = prompt_template | llm | output_parser

    def generate_response(self, theme, update, history):
        recent_history = "\n".join(history[-8:])
        response = self.chain.invoke({
            "name": self.name,
            "leader": self.leader,
            "theme": theme,
            "update": update,
            "history": recent_history,
            "DEBATE_CONTEXT": DEBATE_CONTEXT
        })
        max_length = random.choice([400, 600, 800])
        response_s = response.strip()
        if len(response_s) > max_length:
            response_s = response_s[:max_length] + "..."
        return response_s

# --- Classe Moderator Agent ---
class ModeratorAgent:
    def __init__(self):
        self.chain = moderator_prompt_template | llm | output_parser

    def decide_intervention(self, theme, update, history):
        recent_history = "\n".join(history[-8:])
        response = self.chain.invoke({
            "theme": theme,
            "update": update,
            "history": recent_history,
            "DEBATE_CONTEXT": DEBATE_CONTEXT
        })
        return response.strip()

# --- Classe Debate Simulator ---
class DebateSimulator:
    def __init__(self, theme):
        self.theme = theme
        self.agents = [PartyAgent(name, profile) for name, profile in PARTY_PROFILES.items()]
        self.moderator = ModeratorAgent()
        random.shuffle(self.agents)
        self.history: List[str] = []

    def run_debate(self, updates: List[str]):
        _updates = [f"O tema central deste debate é: {self.theme}.", *updates]
        for index, update in enumerate(_updates):
            current_round_agents = self.agents[:]
            random.shuffle(current_round_agents)
            first_agent = current_round_agents[0]

            if index == 0:
                opening = f"Boa noite e bem-vindos ao nosso debate sobre {self.theme}. {update} Para começar, vou dirigir-me a {first_agent.leader}: qual é a sua leitura inicial desta situação?"
                print(f"\n[Moderador]: 🎙️ {opening}")
                self.history.append(f"[Moderador]: {opening}")
            else:
                mod_intervention = self.moderator.decide_intervention(self.theme, update, self.history)
                if mod_intervention:
                    print(f"\n[Moderador]: {mod_intervention}")
                    self.history.append(f"[Moderador]: {mod_intervention}")

            for agent in current_round_agents:
                if random.random() < 0.85:
                    try:
                        response = agent.generate_response(self.theme, update, self.history)
                        response_line = f"[{agent.name} - {agent.leader}]: {response}"
                        print(f"\n{response_line}")
                        self.history.append(response_line)
                    except Exception as e:
                        error_line = f"[{agent.name} - {agent.leader}]: ⚠️ Erro ao gerar resposta: {e}"
                        print(f"\n{error_line}")
                        self.history.append(error_line)

        closing_message = self.moderator.decide_intervention(self.theme, "Encerramento do debate", self.history)
        if closing_message:
            print(f"\n[Moderador]: {closing_message}")
            self.history.append(f"[Moderador]: {closing_message}")

    def export_history(self, filename="debate_history_realista.txt"):
        try:
            with open(filename, "w", encoding="utf-8") as f:
                for line in self.history:
                    f.write(line + "\n")
            print(f"\n✅ Debate completo exportado para '{filename}'")
        except IOError as e:
            print(f"\n❌ Erro ao exportar histórico para '{filename}': {e}")

# --- Execução Principal ---
if __name__ == "__main__":
    debate_theme = "Impacto das tarifas dos EUA sobre produtos da UE: guerra comercial ou oportunidade para a economia portuguesa?"
    debate_updates = [
        "Os EUA anunciam tarifas de 20% sobre vinho e têxteis europeus.",
        "Confirmação oficial: tarifas entram em vigor na próxima semana.",
        "A UE anuncia retaliação e cria fundo de apoio de 2 mil milhões de euros."
    ]

    debate = DebateSimulator(debate_theme)
    debate.run_debate(debate_updates)
    debate.export_history()



[Moderador]: 🎙️ Boa noite e bem-vindos ao nosso debate sobre Impacto das tarifas dos EUA sobre produtos da UE: guerra comercial ou oportunidade para a economia portuguesa?. O tema central deste debate é: Impacto das tarifas dos EUA sobre produtos da UE: guerra comercial ou oportunidade para a economia portuguesa?. Para começar, vou dirigir-me a Rui Rocha: qual é a sua leitura inicial desta situação?

[BE - Mariana Mortágua]: 🤔 Boa noite. Impacto? É inegável. Oportunidade? Veremos. Mas uma coisa é certa: não podemos encarar isto como se fôssemos meros espectadores. Portugal tem de se preparar e defender os seus interesses.

[PS - Pedro Nuno Santos]: Boa noite. 🇵🇹 A Mariana tem razão num ponto: não podemos ficar de braços cruzados. O impacto existe, e é preciso geri-lo. Agora, dizer que não há oportunidades é que não me parece correto. Há sempre espaço para nos adaptarmos, para procurarmos novos mercados e para reforçarmos a nossa competitividade.

[PSD - Luís Montenegro]: 🇵🇹 Boa noite.

In [44]:
!pip install elevenlabs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.7/413.7 kB 5.6 MB/s eta 0:00:00


In [51]:
!pip install pydub

In [57]:
import os
import uuid

from elevenlabs import VoiceSettings
from elevenlabs.client import ElevenLabs
from pydub import AudioSegment

client = ElevenLabs(
    api_key=None
)

# Mapear personagens para vozes
CHARACTER_VOICE_MAP = {
    "Moderador": "Adam",
    "BE - Mariana Mortágua": "Rachel",
    "PS - Pedro Nuno Santos": "Antoni",
    "PSD - Luís Montenegro": "Josh",
    "Herman José - Diácono Remédios": "Clyde",
    "Chega - André Ventura": "Sam",
    "IL - Rui Rocha": "Elli",
}

def text_to_speech(text: str, voice_id: str) -> str:
    response = client.text_to_speech.convert(
        voice_id="JBFqnCBsd6RMkjVDRZzb",
        optimize_streaming_latency="0",
        output_format="mp3_22050_32",
        text=text,
        model_id="eleven_multilingual_v2",
        voice_settings=VoiceSettings(
            stability=0.3,
            similarity_boost=1.0,
            style=0.0,
            use_speaker_boost=True,
        ),
    )

    save_file_path = f"{uuid.uuid4()}.mp3"
    with open(save_file_path, "wb") as f:
        for chunk in response:
            if chunk:
                f.write(chunk)

    return save_file_path


def parse_script(script: str):
    dialogues = []
    current_speaker = None
    current_text = []

    for line in script.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("[") and "]" in line:
            if current_speaker and current_text:
                dialogues.append((current_speaker, " ".join(current_text)))
                current_text = []

            current_speaker = line.split("]")[0][1:]
            spoken_text = line.split("]:", 1)[1].strip() if ":" in line else ""
            if spoken_text:
                current_text.append(spoken_text)
        else:
            current_text.append(line)

    if current_speaker and current_text:
        dialogues.append((current_speaker, " ".join(current_text)))

    return dialogues


def combine_audio(file_paths, output_file="debate_final.mp3"):
    combined = AudioSegment.empty()
    for file_path in file_paths:
        audio = AudioSegment.from_mp3(file_path)
        combined += audio + AudioSegment.silent(duration=500)  # Pausa entre falas
    combined.export(output_file, format="mp3")
    print(f"Debate completo salvo em: {output_file}")


if __name__ == "__main__":
    # Teu texto do debate aqui
    with open("debate_history_realista.txt", encoding="utf-8") as f:
        script = f.read()

    dialogues = parse_script(script)
    audio_files = []

    for speaker, text in dialogues:
        voice = CHARACTER_VOICE_MAP.get(speaker, "Adam")  # Voz padrão
        print(f"Gerando áudio para {speaker} com voz {voice}...")
        audio_file = text_to_speech(text, voice)
        audio_files.append(audio_file)

    combine_audio(audio_files)


Gerando áudio para Moderador com voz Adam...
Gerando áudio para BE - Mariana Mortágua com voz Rachel...
Gerando áudio para PS - Pedro Nuno Santos com voz Antoni...
Gerando áudio para PSD - Luís Montenegro com voz Josh...
Gerando áudio para Herman José - Diácono Remédios com voz Clyde...
Gerando áudio para Chega - André Ventura com voz Sam...
Gerando áudio para Moderador com voz Adam...
Gerando áudio para PSD - Luís Montenegro com voz Josh...
Gerando áudio para Herman José - Diácono Remédios com voz Clyde...
Gerando áudio para IL - Rui Rocha com voz Elli...
Gerando áudio para PS - Pedro Nuno Santos com voz Antoni...
Gerando áudio para Moderador com voz Adam...
Gerando áudio para BE - Mariana Mortágua com voz Rachel...
Gerando áudio para Herman José - Diácono Remédios com voz Clyde...
Gerando áudio para Chega - André Ventura com voz Sam...
Gerando áudio para IL - Rui Rocha com voz Elli...
Gerando áudio para PS - Pedro Nuno Santos com voz Antoni...
Gerando áudio para Moderador com voz Ada